# 03 — Architektura systemu FURASSIST

**Projekt:** AI-asystowana nawigacja USG | Vet Eye S.A. | BiznesAI 15, ALK  
**Autor:** Marta Kościelna  

## Cel notebooka

Reference implementation całego pipeline'u FURASSIST — od surowego obrazu USG
do tekstowej instrukcji nawigacyjnej dla operatora. Notebook dokumentuje:

1. Ładowanie modelu z HF Model Hub (bez lokalnych plików pośrednich)
2. Post-processing z `ConfidenceThresholder` (próg 0.85)
3. Generowanie instrukcji nawigacyjnych przez `AFASTNavigator`
4. Metryki na trzech zbiorach ewaluacyjnych: val / holdout_easy / holdout_hard
5. Macierz konfuzji i stress test robustności

## Architektura systemu

```
Obraz USG (PIL)
     │
     ▼
eval_transform          Grayscale(3) → Resize(224) → ToTensor → Normalize(ImageNet)
     │
     ▼
TinyUSFM (ViT-Tiny)     5.5M params, backbone pre-trained na 2M obrazach USG
     │                  head: Linear(192 → 4 klas AFAST)
     │                  fine-tuned na koscielnamarta/synthetic-usg-afast-vet
     ▼
softmax (4 wartości)    [P(CC), P(DH), P(HR), P(SR)]
     │
     ▼
ConfidenceThresholder   max(softmax) ≥ 0.85 → klasa  |  < 0.85 → "niepewne"
     │
     ▼
AFASTNavigator          (current_view, views_done) → tekst instrukcji (PL/EN)
```

## Linki

- **Dataset:** [koscielnamarta/synthetic-usg-afast-vet](https://huggingface.co/datasets/koscielnamarta/synthetic-usg-afast-vet)
- **Model:** [koscielnamarta/synthetic-usg-afast-vet-classifier](https://huggingface.co/koscielnamarta/synthetic-usg-afast-vet-classifier)
- **Demo (HF Spaces):** [koscielnamarta/vet-eye-usg-demo](https://huggingface.co/spaces/koscielnamarta/vet-eye-usg-demo)

*Punkt wyjścia dla tego notebooka: [`02b_finetune_synthetic_usg.ipynb`](02b_finetune_synthetic_usg.ipynb) — kod treningu i historia iteracji.*

## 0. Instalacja zależności

**Wymagania:**
- `timm==0.5.4` — TinyUSFM wymaga tej konkretnej wersji (nowsze psują import)
- `huggingface_hub` — pobieranie checkpointu i datasetu
- `scikit-learn` — metryki ewaluacyjne (classification_report, confusion_matrix)

In [ ]:
!pip install -q --upgrade huggingface_hub
!pip install -q timm==0.5.4
!pip install -q scikit-learn matplotlib seaborn

import timm
assert timm.__version__ == '0.5.4', (
    f'Wymagane timm==0.5.4, zainstalowano {timm.__version__}. '
    'Uruchom ponownie komórkę i zrestartuj runtime jeśli Colab pokazuje konflikt.'
)
print(f'timm {timm.__version__} — OK')
print('Zależności zainstalowane.')

## 1. Setup TinyUSFM

TinyUSFM nie jest dostępny jako pakiet PyPI — wymaga klonowania repozytorium
z kodem architektury i dodania go do `sys.path`.

In [ ]:
import os
import sys

TINYUSFM_DIR = '/content/TinyUSFM'

if not os.path.exists(TINYUSFM_DIR):
    os.system(f'git clone https://github.com/MacDunno/TinyUSFM.git {TINYUSFM_DIR}')
    print('TinyUSFM sklonowany.')
else:
    print(f'TinyUSFM już istnieje: {TINYUSFM_DIR}')

if TINYUSFM_DIR not in sys.path:
    sys.path.insert(0, TINYUSFM_DIR)

from model.tinyusfm import TinyUSFM
print('Import TinyUSFM OK.')

## 2. Konfiguracja

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# --- HF identyfikatory ---
HF_MODEL_REPO   = 'koscielnamarta/synthetic-usg-afast-vet-classifier'
HF_DATASET_REPO = 'koscielnamarta/synthetic-usg-afast-vet'
MODEL_CKPT_PATH = 'checkpoints/best_i2.pt'   # I2 partial unfreezing, best val_acc=0.9988

# --- Architektura ---
LABEL_NAMES = ['CC', 'DH', 'HR', 'SR']       # kolejność alfabetyczna (ImageFolder)
NUM_CLASSES = 4
HEAD_DIM    = 192                              # hidden_dim ViT-Tiny
IMG_SIZE    = 224
BATCH_SIZE  = 64

# --- ConfidenceThresholder ---
THRESHOLD     = 0.85
ABSTAIN_LABEL = 'niepewne'
# Uzasadnienie progu 0.85: wartość z Plan_Prac + zweryfikowana w notebook 04.
# Sweep automatyczny na czystym zbiorze val daje optimum=0.50 (model praktycznie
# nigdy nie jest niepewny na syntetyku). Próg 0.85 jest decyzją ekspercką,
# gwarantującą odrzucanie granicznych predykcji w warunkach produkcyjnych.

# --- Device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Klasy AFAST ({NUM_CLASSES}): {LABEL_NAMES}')
print(f'Próg pewności: {THRESHOLD}')

## 3. Ładowanie modelu z HF Model Hub

**Procedura ładowania:**

1. Budujemy architekturę TinyUSFM (ViT-Tiny, bez wag).
2. Podmieniamy domyślną głowicę (`Identity`) na `Linear(192 → 4 klas AFAST)`.
3. Ładujemy `best_i2.pt` z HF — zawiera **pełny** `state_dict` (backbone + głowica)
   zapisany po fine-tuningu w notebook 02b. Nie potrzebujemy osobno inicjalizować
   pretrained backbone z Google Drive, ponieważ `best_i2.pt` nadpisuje wszystkie wagi.

**Uwaga:** `strict=True` (domyślne) — wszystkie klucze muszą pasować.
Jeśli pojawi się błąd kształtu — sprawdź czy `HEAD_DIM=192` i `NUM_CLASSES=4`.

In [ ]:
from huggingface_hub import hf_hub_download


def load_furassist_model(device: torch.device) -> TinyUSFM:
    """Ładuje TinyUSFM z fine-tunedymi wagami AFAST z HF Hub."""
    # Krok 1: architektura z właściwym head
    model = TinyUSFM()
    vit   = model.model                           # VisionTransformer siedzi w model.model
    vit.head = nn.Linear(HEAD_DIM, NUM_CLASSES)   # podmiana Identity → Linear(192, 4)

    # Krok 2: fine-tuned wagi z HF
    ckpt_path = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=MODEL_CKPT_PATH,
    )
    state_dict = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state_dict)             # strict=True: pełny state_dict

    model.to(device)
    model.eval()
    return model


print('Ładowanie modelu z HF...')
MODEL = load_furassist_model(DEVICE)

total_params = sum(p.numel() for p in MODEL.parameters())
print(f'Model: TinyUSFM (ViT-Tiny, fine-tuned AFAST 4 klasy)')
print(f'Parametry: {total_params:,} ({total_params/1e6:.1f}M)')
print(f'Checkpoint: {HF_MODEL_REPO}/{MODEL_CKPT_PATH}')

# Sanity check: forward pass z losowym tensorem
with torch.no_grad():
    _test = MODEL(torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
assert _test.shape == (1, NUM_CLASSES), f'Błędny kształt wyjścia: {_test.shape}'
print('Forward pass OK.')

## 4. Transformacje obrazów

Transformacje muszą być **identyczne** z `eval_transform` z notebook 02b.
Obrazy datasetu to 256×256 grayscale PNG — konwertujemy do RGB (3 kanały)
i normalizujemy ImageNet mean/std (TinyUSFM trenowany na wzorcach USG,
backbone inicjalizowany z ImageNet).

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # grayscale USG → 3 kanały
    transforms.Resize((IMG_SIZE, IMG_SIZE)),        # 256×256 → 224×224
    transforms.ToTensor(),                          # PIL [0, 255] → tensor [0.0, 1.0]
    transforms.Normalize(mean=IMAGENET_MEAN,        # normalizacja ImageNet
                         std=IMAGENET_STD),
])

print('eval_transform zdefiniowany:')
print('  Grayscale(3) → Resize(224) → ToTensor → Normalize(ImageNet)')

## 5. ConfidenceThresholder + AFASTNavigator

Oba moduły są częścią repozytorium projektu (`src/`). Pobieramy je
bezpośrednio z GitHuba (raw URL), żeby notebook był self-contained
i nie wymagał lokalnego klonu repo.

In [ ]:
import urllib.request

SRC_DIR  = '/content/src'
RAW_BASE = 'https://raw.githubusercontent.com/koscielnamarta/vet-eye-ai-usg-demo/main/src'

os.makedirs(SRC_DIR, exist_ok=True)

for module in ['confidence_thresholder.py', 'scripted_instructions.py']:
    dst = os.path.join(SRC_DIR, module)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f'{RAW_BASE}/{module}', dst)
        print(f'  Pobrano: {module}')
    else:
        print(f'  Istnieje: {module}')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from confidence_thresholder import ConfidenceThresholder
from scripted_instructions import AFASTNavigator, AFAST_SCAN_ORDER

# Inicjalizacja z progiem 0.85
thresholder = ConfidenceThresholder(
    threshold=THRESHOLD,
    label_names=LABEL_NAMES,
    abstain_label=ABSTAIN_LABEL,
)
navigator_pl = AFASTNavigator(lang='pl')
navigator_en = AFASTNavigator(lang='en')

print(f'\n{thresholder}')
print(f'AFASTNavigator(lang=pl), kolejność skanowania: {AFAST_SCAN_ORDER}')

## 5b. Demonstracja pipeline'u na przykładowych obrazach

Pobieramy dataset z HF i uruchamiamy pełny pipeline na kilku przykładach —
jeden obraz z każdej klasy, żeby zademonstrować działanie systemu end-to-end.

In [ ]:
from huggingface_hub import snapshot_download
from torchvision.datasets import ImageFolder

# ⚠️ Pobieranie datasetu z HF zajmuje ok. 12–15 minut przy pierwszym uruchomieniu.
# Przy kolejnych uruchomieniach dane są w cache i ładują się natychmiast.
print(f'Pobieranie datasetu: {HF_DATASET_REPO}')
DATA_ROOT = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    local_dir='/content/dataset_afast',
    allow_patterns=['dataset/**'],
    max_workers=2,
)

VAL_DIR          = os.path.join(DATA_ROOT, 'dataset', 'test')
HOLDOUT_DIR      = os.path.join(DATA_ROOT, 'dataset', 'holdout')
HOLDOUT_HARD_DIR = os.path.join(DATA_ROOT, 'dataset', 'synthetic_hard_holdout')

for name, path in [('val', VAL_DIR), ('holdout_easy', HOLDOUT_DIR),
                   ('holdout_hard', HOLDOUT_HARD_DIR)]:
    total = sum(
        len(os.listdir(os.path.join(path, c)))
        for c in os.listdir(path) if os.path.isdir(os.path.join(path, c))
    )
    print(f'  {name:14s}: {total} obrazów')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob


def predict_single(image: Image.Image) -> dict:
    """
    Pełny pipeline dla jednego obrazu.

    Zwraca słownik z: prediction, confidence, probs, abstain.
    """
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(MODEL(tensor), dim=1).squeeze(0).cpu().numpy()
    prediction = thresholder.predict(probs)
    return {
        'prediction':  prediction,
        'confidence':  float(probs.max()),
        'probs':       {l: float(p) for l, p in zip(LABEL_NAMES, probs)},
        'abstain':     prediction == ABSTAIN_LABEL,
    }


# Jeden przykład z każdej klasy (ze zbioru val)
fig, axes = plt.subplots(1, len(LABEL_NAMES), figsize=(14, 4))
fig.suptitle('Pipeline FURASSIST — przykładowe predykcje (zbiór val)', fontsize=12)

for ax, cls in zip(axes, LABEL_NAMES):
    cls_dir = os.path.join(VAL_DIR, cls)
    img_path = sorted(glob.glob(os.path.join(cls_dir, '*.png')))[0]
    img = Image.open(img_path).convert('RGB')

    result = predict_single(img)
    pred   = result['prediction']
    conf   = result['confidence']

    color  = 'green' if pred == cls else ('orange' if result['abstain'] else 'red')
    title  = f'True: {cls}\nPred: {pred} ({conf:.1%})'

    # Instrukcja nawigacyjna (jeśli nie abstain)
    if not result['abstain']:
        instr = navigator_pl.get_instruction_auto(pred, views_done=[])
    else:
        instr = navigator_pl.get_abstain_instruction()

    ax.imshow(img, cmap='gray')
    ax.set_title(title, color=color, fontsize=9, fontweight='bold')
    ax.set_xlabel(f'→ {instr[:55]}…' if len(instr) > 55 else f'→ {instr}',
                  fontsize=7, wrap=True)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/example_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: /content/example_predictions.png')

## 6. Metryki na trzech zbiorach ewaluacyjnych

| Zbiór | Opis | Rozmiar |
|---|---|---|
| `val (test)` | Zbiór walidacyjny z treningu 02b (czyste syntetyki) | 800 |
| `holdout_easy` | Niezależny holdout, ta sama dystrybucja co val (seed=123) | 800 |
| `holdout_hard` | Holdout z trudniejszą dystrybucją (OOD-like, artefakty) | 800 |

**Metryki raportowane z `ConfidenceThresholder(threshold=0.85)`:**
- `accuracy_accepted`: poprawność wśród zaakceptowanych predykcji (model się odezwał)
- `accuracy_overall`: poprawność ogólna (abstain liczymy jako błąd)
- `abstain_rate`: frakcja obrazów odrzuconych przez próg pewności
- `macro_F1`: F1 uśredniony po klasach (standard dla zbalansowanych zbiorów)

In [ ]:
from torch.utils.data import DataLoader
from torch.nn.functional import softmax as torch_softmax


@torch.no_grad()
def collect_predictions(loader) -> tuple[np.ndarray, np.ndarray]:
    """Zbiera probs (N, 4) i labels (N,) dla całego DataLoadera."""
    all_probs, all_labels = [], []
    MODEL.eval()
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE)
        probs  = torch_softmax(MODEL(imgs), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


# Budujemy loadery dla każdego zbioru
splits = {
    'val':          VAL_DIR,
    'holdout_easy': HOLDOUT_DIR,
    'holdout_hard': HOLDOUT_HARD_DIR,
}
loaders   = {}
all_probs = {}
all_labels = {}

for name, path in splits.items():
    ds = ImageFolder(path, transform=eval_transform)
    # Weryfikacja kolejności klas
    assert ds.classes == LABEL_NAMES, f'Błąd klas w {name}: {ds.classes}'
    loaders[name] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=2, pin_memory=True)

print('Zbieranie predykcji...')
for name, loader in loaders.items():
    probs, labels = collect_predictions(loader)
    all_probs[name]  = probs
    all_labels[name] = labels
    baseline_acc = (probs.argmax(axis=1) == labels).mean()
    mean_max     = probs.max(axis=1).mean()
    print(f'  {name:14s}: n={len(labels)} | baseline_acc={baseline_acc:.4f} '
          f'| mean_max_prob={mean_max:.4f}')

In [ ]:
from sklearn.metrics import classification_report

print('=' * 70)
print(f'METRYKI Z CONFIDENCE THRESHOLDING (próg = {THRESHOLD})')
print('=' * 70)

results = {}

for name in splits:
    metrics = thresholder.evaluate(all_probs[name], all_labels[name])
    results[name] = metrics

    # Macro F1 z per_class dict
    macro_f1 = float(np.mean([v['f1'] for v in metrics['per_class'].values()]))

    print(f'\n[{name}]')
    print(f'  Accuracy accepted:  {metrics["accuracy_accepted"]:.4f}')
    print(f'  Accuracy overall:   {metrics["accuracy_overall"]:.4f}')
    print(f'  Abstain rate:       {metrics["abstain_rate"]:.4f}  '
          f'({metrics["n_abstained"]}/{metrics["n_total"]})')
    print(f'  Macro F1:           {macro_f1:.4f}')
    print(f'  Per-klasa F1:       ' +
          ' | '.join(f'{c}: {metrics["per_class"][c]["f1"]:.3f}'
                     for c in LABEL_NAMES))

# Tabela podsumowująca
print('\n' + '=' * 70)
print('TABELA ZBIORCZA')
print(f'{"Zbiór":16s} {"Acc(accepted)":>14} {"Acc(overall)":>13} '
      f'{"Abstain":>8} {"MacroF1":>8}')
print('-' * 70)
for name in splits:
    m  = results[name]
    mf = float(np.mean([v['f1'] for v in m['per_class'].values()]))
    print(f'{name:16s} {m["accuracy_accepted"]:>14.4f} {m["accuracy_overall"]:>13.4f} '
          f'{m["abstain_rate"]:>8.4f} {mf:>8.4f}')
print('=' * 70)

## 7. Confusion matrix

Macierze konfuzji dla modelu **bez thresholdingu** (baseline) — żeby zobaczyć,
które klasy są mylone, niezależnie od decyzji o abstain.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix


def plot_confusion_matrix(labels: np.ndarray, preds: np.ndarray,
                          class_names: list, title: str, out_path: str = None):
    """Rysuje confusion matrix (counts + row-normalized)."""
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, data, sub_title, fmt in [
        (axes[0], cm,      'Counts',                   'd'),
        (axes[1], cm_norm, 'Row-normalized (recall)',  '.2f'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, vmin=0)
        ax.set(xlabel='Predicted', ylabel='True', title=sub_title)
        ax.tick_params(axis='x', rotation=0)

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    if out_path:
        print(f'Zapisano: {out_path}')


for name in splits:
    preds = all_probs[name].argmax(axis=1)
    plot_confusion_matrix(
        labels     = all_labels[name],
        preds      = preds,
        class_names= LABEL_NAMES,
        title      = f'Confusion Matrix — {name} (bez thresholdingu)',
        out_path   = f'/content/confusion_matrix_{name}.png',
    )

## 8. Analiza rozkładu pewności (softmax)

Histogramy `max(softmax)` dla każdego zbioru — weryfikacja decyzji o progu 0.85.
Zielone = predykcje poprawne, czerwone = błędne. Pionowa linia = próg 0.85.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Rozkład max(softmax) na 3 zbiorach (próg = {THRESHOLD})', fontsize=12)

split_colors = {'val': 'steelblue', 'holdout_easy': 'seagreen', 'holdout_hard': 'crimson'}

for ax, name in zip(axes, splits):
    probs  = all_probs[name]
    labels = all_labels[name]
    max_p  = probs.max(axis=1)
    correct = probs.argmax(axis=1) == labels

    ax.hist(max_p[correct],  bins=30, alpha=0.6, color='green',
            label=f'Poprawne ({correct.sum()})', density=True)
    ax.hist(max_p[~correct], bins=30, alpha=0.7, color='red',
            label=f'Błędne ({(~correct).sum()})', density=True)
    ax.axvline(THRESHOLD, color='orange', linestyle='--',
               linewidth=2, label=f'Próg={THRESHOLD}')

    abstain_pct = (max_p < THRESHOLD).mean() * 100
    ax.set_title(f'{name}\n(abstain@{THRESHOLD}: {abstain_pct:.1f}%)', fontsize=9)
    ax.set(xlabel='max(softmax)', ylabel='Gęstość')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('/content/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: /content/confidence_distribution.png')

## 9. Stress test — robustność na degradację obrazu

Sprawdzamy jak accuracy spada pod wpływem szumu Gaussowskiego,
rozmycia i częściowej zasłony obrazu.

**Oczekiwany wynik:** hard holdout degraduje znacznie szybciej niż easy —
to argument za utrzymaniem progu 0.85 w warunkach produkcyjnych.

In [ ]:
from PIL import ImageFilter
from torch.utils.data import Dataset


class StressedImageFolder(ImageFolder):
    """ImageFolder z testem degradacji obrazu (szum / rozmycie / zasłona)."""

    def __init__(self, root, transform, noise_std=0.0,
                 blur_radius=0.0, partial_mask=False):
        super().__init__(root, transform=None)
        self.final_transform = transform
        self.noise_std    = noise_std
        self.blur_radius  = blur_radius
        self.partial_mask = partial_mask

    def __getitem__(self, index):
        path, target = self.samples[index]
        img = self.loader(path)

        if self.blur_radius > 0:
            img = img.filter(ImageFilter.GaussianBlur(radius=self.blur_radius))
        if self.noise_std > 0:
            arr = np.array(img).astype(np.float32)
            arr += np.random.randn(*arr.shape) * 255 * self.noise_std
            img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
        if self.partial_mask:
            # Zasłona dolnej połowy obrazu (częsty artefakt w USG)
            arr = np.array(img).astype(np.float32)
            arr[arr.shape[0] // 2:, :] *= 0.15
            img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

        if self.final_transform:
            img = self.final_transform(img)
        return img, target


@torch.no_grad()
def eval_stressed(base_dir: str, **stress_kwargs) -> float:
    ds     = StressedImageFolder(base_dir, eval_transform, **stress_kwargs)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    MODEL.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        preds    = MODEL(imgs.to(DEVICE)).argmax(dim=1).cpu()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total


stress_configs = [
    {'label': 'Clean',            'noise_std': 0.00, 'blur_radius': 0.0, 'partial_mask': False},
    {'label': 'Noise 5%',         'noise_std': 0.05, 'blur_radius': 0.0, 'partial_mask': False},
    {'label': 'Noise 10% + blur', 'noise_std': 0.10, 'blur_radius': 1.0, 'partial_mask': False},
    {'label': 'Noise 20% + blur', 'noise_std': 0.20, 'blur_radius': 2.0, 'partial_mask': False},
    {'label': 'Partial mask',     'noise_std': 0.00, 'blur_radius': 0.0, 'partial_mask': True},
]

stress_results = {'holdout_easy': {}, 'holdout_hard': {}}
holdout_dirs   = {'holdout_easy': HOLDOUT_DIR, 'holdout_hard': HOLDOUT_HARD_DIR}

print('Stress test — accuracy przy degradacji obrazu:')
for cfg in stress_configs:
    label = cfg['label']
    kw    = {k: v for k, v in cfg.items() if k != 'label'}
    for name, path in holdout_dirs.items():
        acc = eval_stressed(path, **kw)
        stress_results[name][label] = acc
    print(f'  {label:22s}: easy={stress_results["holdout_easy"][label]:.4f}  '
          f'hard={stress_results["holdout_hard"][label]:.4f}')

In [ ]:
labels_x = [c['label'] for c in stress_configs]
x = np.arange(len(labels_x))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color) in enumerate([('holdout_easy', '#3498db'), ('holdout_hard', '#e67e22')]):
    accs   = [stress_results[name][l] for l in labels_x]
    offset = (i - 0.5) * width
    bars   = ax.bar(x + offset, accs, width, label=name, color=color, edgecolor='white')
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f'{acc:.2f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels_x, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.axhline(0.25, color='red', ls='--', alpha=0.5, label='Random (4 klasy)')
ax.set(ylabel='Accuracy', title='Robustność — easy vs hard holdout (degradacja obrazu)')
ax.legend()
plt.tight_layout()
plt.savefig('/content/stress_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: /content/stress_test.png')

## 10. Podsumowanie

Podsumowanie wyników dla sekcji POC pracy końcowej.

In [ ]:
print('=' * 70)
print('PODSUMOWANIE — FURASSIST PIPELINE')
print('=' * 70)
print()
print('MODEL')
print(f'  Architektura : TinyUSFM (ViT-Tiny, 5.5M parametrów)')
print(f'  Fine-tuning  : dwuetapowy (I1 linear probing + I2 partial unfreezing 1 bloku)')
print(f'  Checkpoint   : {HF_MODEL_REPO}/{MODEL_CKPT_PATH}')
print(f'  Klasy        : {LABEL_NAMES}')
print()
print('POST-PROCESSING')
print(f'  Próg pewności: {THRESHOLD} (ekspercki, z Plan_Prac)')
print(f'  Abstain label: "{ABSTAIN_LABEL}" — model odmawia predykcji poniżej progu')
print()
print('WYNIKI (z ConfidenceThresholder, próg=0.85)')
print(f'{"Zbiór":16s} {"Acc(accepted)":>14} {"Acc(overall)":>13} '
      f'{"Abstain":>9} {"MacroF1":>8}')
print('-' * 70)
for name in splits:
    m  = results[name]
    mf = float(np.mean([v['f1'] for v in m['per_class'].values()]))
    print(f'{name:16s} {m["accuracy_accepted"]:>14.4f} {m["accuracy_overall"]:>13.4f} '
          f'{m["abstain_rate"]:>9.4f} {mf:>8.4f}')
print('=' * 70)
print()
print('OGRANICZENIA')
print('  - Model trenowany wyłącznie na danych syntetycznych.')
print('  - Nie przeszedł walidacji klinicznej — nie nadaje się do diagnostyki.')
print('  - Hard holdout (OOD-like) pokazuje istotny spadek jakości:')
m_hard = results['holdout_hard']
print(f'    acc_accepted={m_hard["accuracy_accepted"]:.4f}, '
      f'abstain_rate={m_hard["abstain_rate"]:.4f}')
print()
print('LINKI')
print(f'  Dataset : https://huggingface.co/datasets/{HF_DATASET_REPO}')
print(f'  Model   : https://huggingface.co/{HF_MODEL_REPO}')
print(f'  Demo    : https://huggingface.co/spaces/koscielnamarta/vet-eye-usg-demo')
print(f'  GitHub  : https://github.com/koscielnamarta/vet-eye-ai-usg-demo')